# OCR-Free Model Comparison

## Setup

In [4]:
# Import Libraries

import ollama
import json
import re
import time
from rapidfuzz import fuzz

### Config

In [5]:
# Models
MODELS = ["qwen2.5vl:3b", "gemma4:e2b"]

In [6]:
# Add this as a temporary cell before Cell 10
import requests

try:
    response = requests.get("http://localhost:11434")
    print("✅ Ollama server is running")
except Exception as e:
    print(f"❌ Ollama server not reachable: {e}")

✅ Ollama server is running


### Test Cases

In [7]:
# Test Cases
TEST_CASES = [
    {
        "image": "../data/receipt/bon-solaria.jpeg",
        "ground_truth": {
            "items": [
                {
                    "name": "Siomay",
                    "quantity": 2,
                    "price": 20002
                },
                {
                    "name": "Air Mineral Botol",
                    "quantity": 1,
                    "price": 10001
                },
                {
                    "name": "Es Teh Manis",
                    "quantity": 1,
                    "price": 8183 
                },
                {
                    "name": "Kwetiau Sapi Siram",
                    "quantity": 1,
                    "price": 44546
                },
                {
                    "name": "Kwetiau Seafood Goreng",
                    "quantity": 1,
                    "price": 46364
                }
            ],
            "charges": [
                {
                    "name": "PB1 10%",
                    "amount": 12910
                },
                {
                    "name": "Rounding",
                    "amount": -6
                }
            ],
            "currency": "IDR"
        }
    },
    {
        "image": "../data/receipt/bon-paul-bakery.jpg",
        "ground_truth": {
            "items": [
                {
                    "name": "Cappucino",
                    "quantity": 1,
                    "price": 45000
                },
                {
                    "name": "Cappucino",
                    "quantity": 1,
                    "price": 45000
                },
                {
                    "name": "Hazelnut Cappucino",
                    "quantity": 1,
                    "price": 50000 
                },
                {
                    "name": "Grn Apple Celery Jui",
                    "quantity": 1,
                    "price": 60000
                },
                {
                    "name": "Croissant Cheese",
                    "quantity": 1,
                    "price": 36000
                },
                {
                    "name": "Hazelnut Cappucino",
                    "quantity": 1,
                    "price": 50000 
                }
            ],
            "charges": [
                {
                    "name": "Service",
                    "amount": 14300
                },
                {
                    "name": "PB1",
                    "amount": 30030
                }
            ],
            "currency": "IDR"
        }
    },
    {
        "image": "../data/receipt/bon-jco.jpg",
        "ground_truth": {
            "items": [
                {
                    "name": "JPops 2 Dzn A",
                    "quantity": 1,
                    "price": 48000
                }
            ],
            "charges": [],
            "currency": "IDR"
        }
    },
    {
        "image": "../data/receipt/bon-hokben.jpg",
        "ground_truth": {
            "items": [
                {
                    "name": "Beef Teriyaki RB",
                    "quantity": 2,
                    "price": 65456
                },
                {
                    "name": "Rice Only RB",
                    "quantity": 2,
                    "price": 10910
                },
                {
                    "name": "Ogura RB",
                    "quantity": 1,
                    "price": 9091
                },
                {
                    "name": "Cold Ocha",
                    "quantity": 2,
                    "price": 10910
                }
            ],
            "charges": [
                {
                    "name": "PJK Resto 10%",
                    "amount": 9637
                },
                {
                    "name": "Pembulatan",
                    "amount": 4
                }
            ],
            "currency": "IDR"
        }
    },
]

| # | Receipt | Unique Challenge |
|---|---|---|
| 1 | Solaria | Two-line item, negative rounding |
| 2 | Paul Bakery | Duplicate entries, no quantity |
| 3 | JCO | Package with sub-items, PB1 included |
| 4 | HokBen | Blurry image, Pembulatan |

## Benchmarking

### Create Benchmarking Building Blocks

In [8]:
SYSTEM_PROMPT = """
You are a receipt parser. Extract only the relevant information from the receipt image.

ules:
- An item is anything that was ordered (food, drinks, or services) with a price
- A charge is any financial adjustment such as tax, service charge, or discount
- Ignore everything else: restaurant name, address, taglines, cashier info,
  loyalty points, table numbers, and any text without a clear price
- If the same item appears multiple times as separate lines, list each one separately
- If a line has no price and is at the same indentation level as the 
  previous item, consolidate it into the previous item's name
- If a line has no price and is indented beneath the previous item, 
  it is a sub-item description — ignore it

Return ONLY a valid JSON object with no explanation or markdown. Use this exact schema:
{
  "items": [
    {"name": "item name", "quantity": 1, "price": 0.00}
  ],
  "charges": [
    {"name": "charge name", "amount": 0.00}
  ],
  "currency": "IDR"
}

Example:
Input: A receipt with nasi goreng, es teh, and 11% tax
Output:
{
  "items": [
    {"name": "Nasi Goreng", "quantity": 1, "price": 45000.00},
    {"name": "Es Teh", "quantity": 2, "price": 16000.00}
  ],
  "charges": [
    {"name": "PB 10%", "amount": 6710.00}
  ],
  "currency": "IDR"
}
"""

In [16]:
import base64
from io import BytesIO
from PIL import Image

def resize_image(image_path: str, max_size: int = 1024) -> str:
    """Resize image and return as base64 string."""
    img = Image.open(image_path)
    img.thumbnail((max_size, max_size))
    buffer = BytesIO()
    img.save(buffer, format=img.format or "JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

def clean_json_output(raw: str) -> dict:
    """Strip markdown fences, remove thousands separators, and parse JSON."""
    # Strip markdown fences
    cleaned = re.sub(r"```json|```", "", raw).strip()
    # Remove thousands separators in numbers e.g. 9,637 → 9637
    cleaned = re.sub(r'(\d),(\d{3})', r'\1\2', cleaned)
    return json.loads(cleaned)

def parse_receipt(image_input: str, model: str) -> dict:
    """Send receipt image to VLM and return parsed JSON."""
    try:
        image_input = resize_image(image_input)  # 👈 resize + convert to base64
        
        response = ollama.chat(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": "Here is the receipt.",
                    "images": [image_input]
                }
            ]
        )
        raw = response["message"]["content"]
        return clean_json_output(raw)

    except json.JSONDecodeError:
        return {"error": "Model returned invalid JSON", "raw": raw}
    except Exception as e:
        return {"error": str(e)}

In [10]:
# Function to compare predicted output against ground truth."""
def score_results(predicted: dict, ground_truth: dict) -> dict:

    # --- Score items ---
    name_scores = []
    price_matches = []

    for exp in ground_truth["items"]:
        best_name_score = 0
        best_price_match = False

        for pred in predicted.get("items", []):
            name_score = fuzz.ratio(
                pred["name"].lower(),
                exp["name"].lower()
            ) / 100
            if name_score > best_name_score:
                best_name_score = name_score
                best_price_match = (pred["price"] == exp["price"])

        name_scores.append(best_name_score)
        price_matches.append(best_price_match)

    # --- Score charges ---
    charge_name_scores = []
    charge_amount_matches = []

    for exp in ground_truth["charges"]:
        best_name_score = 0
        best_amount_match = False

        for pred in predicted.get("charges", []):
            name_score = fuzz.ratio(
                pred["name"].lower(),
                exp["name"].lower()
            ) / 100
            if name_score > best_name_score:
                best_name_score = name_score
                best_amount_match = (pred["amount"] == exp["amount"])

        charge_name_scores.append(best_name_score)
        charge_amount_matches.append(best_amount_match)

    return {
        "avg_item_name_score": sum(name_scores) / len(name_scores) if name_scores else 0,
        "item_price_accuracy": sum(price_matches) / len(price_matches) if price_matches else 0,
        "avg_charge_name_score": sum(charge_name_scores) / len(charge_name_scores) if charge_name_scores else 1.0,
        "charge_amount_accuracy": sum(charge_amount_matches) / len(charge_amount_matches) if charge_amount_matches else 1.0,
    }

In [11]:
# Function to compare two VLMs
from tqdm import tqdm

def benchmark(model: str) -> dict:
    time_scores = []
    item_name_scores = []
    item_price_accuracies = []
    charge_name_scores = []
    charge_amount_accuracies = []

    for test in tqdm(TEST_CASES, desc=f"🔍 {model}", unit="receipt"):
        start = time.time()
        predicted = parse_receipt(test["image"], model)
        end = time.time()
        time_scores.append(end - start)

        if "error" in predicted:
            tqdm.write(f"  ⚠️  Error on {test['image'].split('/')[-1]}: {predicted['error']}")
            continue

        scores = score_results(predicted, test["ground_truth"])
        item_name_scores.append(scores["avg_item_name_score"])
        item_price_accuracies.append(scores["item_price_accuracy"])
        charge_name_scores.append(scores["avg_charge_name_score"])
        charge_amount_accuracies.append(scores["charge_amount_accuracy"])

    return {
        "model": model,
        "avg_time": sum(time_scores) / len(time_scores),
        "avg_item_name_score": sum(item_name_scores) / len(item_name_scores) if item_name_scores else 0,
        "avg_item_price_accuracy": sum(item_price_accuracies) / len(item_price_accuracies) if item_price_accuracies else 0,
        "avg_charge_name_score": sum(charge_name_scores) / len(charge_name_scores) if charge_name_scores else 0,
        "avg_charge_amount_accuracy": sum(charge_amount_accuracies) / len(charge_amount_accuracies) if charge_amount_accuracies else 0,
    }

### Run Bulk Benchmarking

In [18]:
all_results = []

for model in MODELS:
    print(f"\n🔍 Benchmarking: {model}")
    results = benchmark(model)
    all_results.append(results)


🔍 Benchmarking: qwen2.5vl:3b


🔍 qwen2.5vl:3b: 100%|██████████| 4/4 [00:19<00:00,  4.89s/receipt]



🔍 Benchmarking: gemma4:e2b


🔍 gemma4:e2b: 100%|██████████| 4/4 [01:00<00:00, 15.14s/receipt]


In [19]:
print("\n" + "="*60)
print("BENCHMARK RESULTS")
print("="*60)

for r in all_results:
    print(f"\nModel: {r['model']}")
    print(f"  Avg response time:          {r['avg_time']:.2f}s")
    print(f"  Avg item name score:         {r['avg_item_name_score']:.2f}")
    print(f"  Avg item price accuracy:     {r['avg_item_price_accuracy']*100:.1f}%")
    print(f"  Avg charge name score:       {r['avg_charge_name_score']:.2f}")
    print(f"  Avg charge amount accuracy:  {r['avg_charge_amount_accuracy']*100:.1f}%")

print("\n" + "="*60)


BENCHMARK RESULTS

Model: qwen2.5vl:3b
  Avg response time:          4.89s
  Avg item name score:         0.86
  Avg item price accuracy:     70.0%
  Avg charge name score:       0.71
  Avg charge amount accuracy:  75.0%

Model: gemma4:e2b
  Avg response time:          15.14s
  Avg item name score:         0.69
  Avg item price accuracy:     70.8%
  Avg charge name score:       0.63
  Avg charge amount accuracy:  50.0%



### Run Benchmarking for Qwen2.5VL

In [20]:
# Pick model and receipt
model = MODELS[0]   # qwen2.5vl:3b
test = TEST_CASES[0]  # Solaria

predicted = parse_receipt(test["image"], model)

# Print benchmarking result
print("PREDICTED:")
print(json.dumps(predicted, indent=2))

print("\nGROUND TRUTH:")
print(json.dumps(test["ground_truth"], indent=2))

scores = score_results(predicted, test["ground_truth"])
print("\nSCORES:")
print(json.dumps(scores, indent=2))

PREDICTED:
{
  "items": [
    {
      "name": "Siomay",
      "quantity": 2,
      "price": 20000.0
    },
    {
      "name": "Air Mineral Botol",
      "quantity": 1,
      "price": 10001.0
    },
    {
      "name": "Es Teh Manis",
      "quantity": 1,
      "price": 8183.0
    },
    {
      "name": "Kwetiau Sapi Siram",
      "quantity": 1,
      "price": 44546.0
    },
    {
      "name": "Kwetiau Seafood",
      "quantity": 1,
      "price": 46364.0
    },
    {
      "name": "Goreng",
      "quantity": 1,
      "price": 129096.0
    }
  ],
  "charges": [
    {
      "name": "PB 10%",
      "amount": 12910.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "Siomay",
      "quantity": 2,
      "price": 20002
    },
    {
      "name": "Air Mineral Botol",
      "quantity": 1,
      "price": 10001
    },
    {
      "name": "Es Teh Manis",
      "quantity": 1,
      "price": 8183
    },
    {
      "name": "Kwetiau Sapi Siram",
      "quantity": 1

In [21]:
# Pick model and receipt
model = MODELS[0]   # qwen2.5vl:3b
test = TEST_CASES[1]  # Paul Bakery

predicted = parse_receipt(test["image"], model)

# Print benchmarking result
print("PREDICTED:")
print(json.dumps(predicted, indent=2))

print("\nGROUND TRUTH:")
print(json.dumps(test["ground_truth"], indent=2))

scores = score_results(predicted, test["ground_truth"])
print("\nSCORES:")
print(json.dumps(scores, indent=2))

PREDICTED:
{
  "items": [
    {
      "name": "Cappuccino",
      "quantity": 1,
      "price": 45000.0
    },
    {
      "name": "Cappuccino",
      "quantity": 1,
      "price": 45000.0
    },
    {
      "name": "Hazelnut Cappuccino",
      "quantity": 1,
      "price": 50000.0
    },
    {
      "name": "Grn Apple Celery Jui",
      "quantity": 1,
      "price": 60000.0
    },
    {
      "name": "Croissant Cheese",
      "quantity": 1,
      "price": 36000.0
    },
    {
      "name": "Hazelnut Cappuccino",
      "quantity": 1,
      "price": 50000.0
    }
  ],
  "charges": [
    {
      "name": "Service",
      "amount": 14300.0
    },
    {
      "name": "PB1",
      "amount": 30030.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "Cappucino",
      "quantity": 1,
      "price": 45000
    },
    {
      "name": "Cappucino",
      "quantity": 1,
      "price": 45000
    },
    {
      "name": "Hazelnut Cappucino",
      "quantity": 1,
      "p

In [22]:
# Pick model and receipt
model = MODELS[0]   # qwen2.5vl:3b
test = TEST_CASES[2]  # Paul Bakery

predicted = parse_receipt(test["image"], model)

# Print benchmarking result
print("PREDICTED:")
print(json.dumps(predicted, indent=2))

print("\nGROUND TRUTH:")
print(json.dumps(test["ground_truth"], indent=2))

scores = score_results(predicted, test["ground_truth"])
print("\nSCORES:")
print(json.dumps(scores, indent=2))

PREDICTED:
{
  "items": [
    {
      "name": "JPOps 2 Den A Box",
      "quantity": 1,
      "price": 48000.0
    }
  ],
  "charges": [
    {
      "name": "PB 10%",
      "amount": 4800.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "JPops 2 Dzn A",
      "quantity": 1,
      "price": 48000
    }
  ],
  "charges": [],
  "currency": "IDR"
}

SCORES:
{
  "avg_item_name_score": 0.8,
  "item_price_accuracy": 1.0,
  "avg_charge_name_score": 1.0,
  "charge_amount_accuracy": 1.0
}


In [23]:
# Pick model and receipt
model = MODELS[0]   # qwen2.5vl:3b
test = TEST_CASES[3]  # Hokben

predicted = parse_receipt(test["image"], model)

# Print benchmarking result
print("PREDICTED:")
print(json.dumps(predicted, indent=2))

print("\nGROUND TRUTH:")
print(json.dumps(test["ground_truth"], indent=2))

scores = score_results(predicted, test["ground_truth"])
print("\nSCORES:")
print(json.dumps(scores, indent=2))

PREDICTED:
{
  "items": [
    {
      "name": "Rice Osa",
      "quantity": 1,
      "price": 0.0
    },
    {
      "name": "Yaki RB",
      "quantity": 1,
      "price": 0.0
    },
    {
      "name": "BEEF TERI",
      "quantity": 1,
      "price": 0.0
    },
    {
      "name": "GOLD OCHA",
      "quantity": 1,
      "price": 0.0
    }
  ],
  "charges": [
    {
      "name": "PB 10%",
      "amount": 9637.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "Beef Teriyaki RB",
      "quantity": 2,
      "price": 65456
    },
    {
      "name": "Rice Only RB",
      "quantity": 2,
      "price": 10910
    },
    {
      "name": "Ogura RB",
      "quantity": 1,
      "price": 9091
    },
    {
      "name": "Cold Ocha",
      "quantity": 2,
      "price": 10910
    }
  ],
  "charges": [
    {
      "name": "PJK Resto 10%",
      "amount": 9637
    },
    {
      "name": "Pembulatan",
      "amount": 4
    }
  ],
  "currency": "IDR"
}

SCORES:
{
  "avg

### Run Benchmarking for Gemma4

In [24]:
# Pick model and receipt
model = MODELS[1]   # gemma4:e2b
test = TEST_CASES[0]  # Solaria

predicted = parse_receipt(test["image"], model)

# Print benchmarking result
print("PREDICTED:")
print(json.dumps(predicted, indent=2))

print("\nGROUND TRUTH:")
print(json.dumps(test["ground_truth"], indent=2))

scores = score_results(predicted, test["ground_truth"])
print("\nSCORES:")
print(json.dumps(scores, indent=2))

PREDICTED:
{
  "items": [
    {
      "name": "Kwetiau Seafood",
      "quantity": 1,
      "price": 129096.0
    },
    {
      "name": "Goreng",
      "quantity": 1,
      "price": 12910.0
    },
    {
      "name": "Siomay",
      "quantity": 1,
      "price": 20002.0
    },
    {
      "name": "Air Mineral Botol",
      "quantity": 1,
      "price": 10001.0
    },
    {
      "name": "Es Teh Manis",
      "quantity": 1,
      "price": 8183.0
    },
    {
      "name": "Kwetiau Sapi Siram",
      "quantity": 1,
      "price": 44546.0
    },
    {
      "name": "Kwetiau Seafood",
      "quantity": 1,
      "price": 46364.0
    }
  ],
  "charges": [
    {
      "name": "Pb1 10%",
      "amount": 0.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "Siomay",
      "quantity": 2,
      "price": 20002
    },
    {
      "name": "Air Mineral Botol",
      "quantity": 1,
      "price": 10001
    },
    {
      "name": "Es Teh Manis",
      "quantity": 1,
 

In [25]:
# Pick model and receipt
model = MODELS[1]   # gemma4:e2b
test = TEST_CASES[1]  # JCO

predicted = parse_receipt(test["image"], model)

# Print benchmarking result
print("PREDICTED:")
print(json.dumps(predicted, indent=2))

print("\nGROUND TRUTH:")
print(json.dumps(test["ground_truth"], indent=2))

scores = score_results(predicted, test["ground_truth"])
print("\nSCORES:")
print(json.dumps(scores, indent=2))

PREDICTED:
{
  "items": [
    {
      "name": "CAPPUCCINO",
      "quantity": 1,
      "price": 45000.0
    },
    {
      "name": "HAZELNUT CAPPUCCINO",
      "quantity": 1,
      "price": 50000.0
    },
    {
      "name": "HAZELNUT",
      "quantity": 1,
      "price": 60000.0
    },
    {
      "name": "CAPPUCCINO",
      "quantity": 1,
      "price": 36000.0
    },
    {
      "name": "HAZELNUT CAPPUCCINO",
      "quantity": 1,
      "price": 50000.0
    }
  ],
  "charges": [
    {
      "name": "Taxable Amount",
      "amount": 300300.0
    },
    {
      "name": "Tax Amount",
      "amount": 30030.0
    },
    {
      "name": "Cash Given",
      "amount": 350000.0
    },
    {
      "name": "Change",
      "amount": 19670.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "Cappucino",
      "quantity": 1,
      "price": 45000
    },
    {
      "name": "Cappucino",
      "quantity": 1,
      "price": 45000
    },
    {
      "name": "Hazelnut Ca

In [27]:
# Pick model and receipt
model = MODELS[1]   # gemma4:e2b
test = TEST_CASES[2]  # Paul Bakery

predicted = parse_receipt(test["image"], model)

# Print benchmarking result
print("PREDICTED:")
print(json.dumps(predicted, indent=2))

print("\nGROUND TRUTH:")
print(json.dumps(test["ground_truth"], indent=2))

scores = score_results(predicted, test["ground_truth"])
print("\nSCORES:")
print(json.dumps(scores, indent=2))

PREDICTED:
{
  "items": [],
  "charges": [
    {
      "name": "Total",
      "amount": 48000.0
    },
    {
      "name": "Total",
      "amount": 50000.0
    },
    {
      "name": "Payment",
      "amount": 50000.0
    },
    {
      "name": "Change",
      "amount": 2000.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "JPops 2 Dzn A",
      "quantity": 1,
      "price": 48000
    }
  ],
  "charges": [],
  "currency": "IDR"
}

SCORES:
{
  "avg_item_name_score": 0.0,
  "item_price_accuracy": 0.0,
  "avg_charge_name_score": 1.0,
  "charge_amount_accuracy": 1.0
}


In [28]:
# Pick model and receipt
model = MODELS[1]   # gemma4:e2b
test = TEST_CASES[3]  # Hokben

predicted = parse_receipt(test["image"], model)

# Print benchmarking result
print("PREDICTED:")
print(json.dumps(predicted, indent=2))

print("\nGROUND TRUTH:")
print(json.dumps(test["ground_truth"], indent=2))

scores = score_results(predicted, test["ground_truth"])
print("\nSCORES:")
print(json.dumps(scores, indent=2))

PREDICTED:
{
  "items": [
    {
      "name": "BEEF TERIYAKI RB",
      "quantity": 2,
      "price": 65456.0
    },
    {
      "name": "RICE ONLY RB",
      "quantity": 2,
      "price": 10910.0
    },
    {
      "name": "OGURA RB",
      "quantity": 1,
      "price": 9091.0
    },
    {
      "name": "COLD OCHA",
      "quantity": 2,
      "price": 10910.0
    }
  ],
  "charges": [
    {
      "name": "SUB TOTAL",
      "amount": 96367.0
    },
    {
      "name": "TOTAL",
      "amount": 106000.0
    }
  ],
  "currency": "IDR"
}

GROUND TRUTH:
{
  "items": [
    {
      "name": "Beef Teriyaki RB",
      "quantity": 2,
      "price": 65456
    },
    {
      "name": "Rice Only RB",
      "quantity": 2,
      "price": 10910
    },
    {
      "name": "Ogura RB",
      "quantity": 1,
      "price": 9091
    },
    {
      "name": "Cold Ocha",
      "quantity": 2,
      "price": 10910
    }
  ],
  "charges": [
    {
      "name": "PJK Resto 10%",
      "amount": 9637
    },
    {
    